# LLIMONIIE step-by-step test notebook

This notebook aligns the local repository assets with the reference workflow in `leonardoPiano/LLIMONIE` so you can test the paper approach on your own dataset in stages.

## Workflow mapping

| Reference stage | Local repo asset | Purpose |
| --- | --- | --- |
| Dataset cleaning | `/home/runner/work/Facts_extraction/Facts_extraction/scripts/01_parse_and_clean.py` | Normalize raw articles into JSONL with `title`, `summary`, `content`, and metadata. |
| ONER prompt | `/home/runner/work/Facts_extraction/Facts_extraction/PromptTemplates/prompt_ner.txt` | Extract entities with `[specific, hypernym]` typing. |
| ORE prompt | `/home/runner/work/Facts_extraction/Facts_extraction/PromptTemplates/prompt_re_fr.txt` | Extract short factual relations from known entities. |
| JOINT prompt | `/home/runner/work/Facts_extraction/Facts_extraction/PromptTemplates/prompt_joint_fr.txt` | Extract entities and triplets in one pass. |
| Step-by-step audit | This notebook | Run a small sample before scaling to the full dataset. |


In [ ]:
!pip -q install transformers accelerate sentencepiece bitsandbytes


In [ ]:
from pathlib import Path
import json
import subprocess

def resolve_repo_root() -> Path:
    candidates = [
        Path('/home/runner/work/Facts_extraction/Facts_extraction'),
        Path('/content/Facts_extraction'),
        Path.cwd(),
    ]
    for candidate in candidates:
        if (candidate / 'PromptTemplates').exists() and (candidate / 'README.md').exists():
            return candidate
    raise FileNotFoundError('Could not locate the repository root.')

REPO_ROOT = resolve_repo_root()
PROMPT_DIR = REPO_ROOT / 'PromptTemplates'
CLEAN_SCRIPT = REPO_ROOT / 'scripts' / '01_parse_and_clean.py'

RAW_DATA_PATH = Path('/content/drive/MyDrive/YOUR_DATASET_FOLDER')
CLEAN_DATA_PATH = REPO_ROOT / 'out' / 'articles_clean.jsonl'
MODEL_ID = 'meta-llama/Meta-Llama-3-8B-Instruct'
MAX_NEW_TOKENS = 512
SAMPLE_INDEX = 0

print('Repo root:', REPO_ROOT)
print('Raw data path:', RAW_DATA_PATH)


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    print('Skipping Google Drive mount outside Colab.')


## 1) Clean your dataset

The reference repository expects normalized JSON examples. This repo provides a cleaning script you can point at one JSON/JSONL file or a directory of exports.


In [ ]:
cmd = [
    'python', str(CLEAN_SCRIPT), str(RAW_DATA_PATH),
    '--output', str(CLEAN_DATA_PATH),
    '--min-content-length', '200',
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
def load_jsonl(path: Path):
    with path.open('r', encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

articles = load_jsonl(CLEAN_DATA_PATH)
sample = articles[SAMPLE_INDEX]
sample


## 2) Load the repo prompts

These prompts are the local equivalents of the instruction templates used in the reference implementation.


In [ ]:
def load_prompt(name: str) -> str:
    return (PROMPT_DIR / name).read_text(encoding='utf-8')

prompt_ner = load_prompt('prompt_ner.txt')
prompt_re = load_prompt('prompt_re_fr.txt')
prompt_joint = load_prompt('prompt_joint_fr.txt')

print(prompt_ner)


import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if torch.cuda.is_available():
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    model_kwargs = {'torch_dtype': dtype, 'device_map': 'auto'}
else:
    model_kwargs = {'torch_dtype': torch.float32}

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    **model_kwargs,
)


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map='auto'
)


In [ ]:
def render_prompt(template: str, text: str, entities_json: str | None = None) -> str:
    prompt = template.replace('{{TEXT}}', text)
    if entities_json is not None:
        prompt = prompt.replace('{{ENTITIES_JSON}}', entities_json)
    return prompt

def generate_from_instruction(instruction: str) -> str:
    messages = [
        {'role': 'user', 'content': instruction},
    ]
    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors='pt'
    ).to(model.device)
    output = model.generate(
        input_ids,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        temperature=0.0,
        pad_token_id=tokenizer.pad_token_id,
    )
    response = output[0][input_ids.shape[-1]:]
    return tokenizer.decode(response, skip_special_tokens=True).strip()


## 4) Stage test: NER → RE → JOINT

This mirrors the paper flow at a notebook scale so you can inspect each stage on your own sample.


In [ ]:
text = sample.get('content') or sample.get('summary') or sample.get('title', '')
ner_instruction = render_prompt(prompt_ner, text)
ner_output = generate_from_instruction(ner_instruction)
print(ner_output)


In [ ]:
re_instruction = render_prompt(prompt_re, text, ner_output)
re_output = generate_from_instruction(re_instruction)
print(re_output)


In [ ]:
joint_instruction = render_prompt(prompt_joint, text)
joint_output = generate_from_instruction(joint_instruction)
print(joint_output)


## 5) Save the audited sample

Use this output format to compare prompt variants before scaling to the full dataset.


In [ ]:
audit_record = {
    'id': sample.get('id'),
    'title': sample.get('title'),
    'text': text,
    'ner_output': ner_output,
    're_output': re_output,
    'joint_output': joint_output,
}

audit_path = REPO_ROOT / 'out' / 'step_by_step_sample.json'
audit_path.parent.mkdir(parents=True, exist_ok=True)
audit_path.write_text(json.dumps(audit_record, ensure_ascii=False, indent=2), encoding='utf-8')
audit_path


## Notes

- This notebook covers the step-by-step inference workflow only.
- The reference repo also contains training helpers (`train.py`, `utils.py`, `instruct_predict.py`), which you can port later once your dataset and prompts are stable.
- For larger tests, iterate over `articles` and store one JSON result per document.
